создание новых признаков для улучшения предсказаний моделей

ошибки лучшей модели(катбуста)

In [107]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

X_tr = pd.read_csv('../data/X_tr.csv')
X_test = pd.read_csv('../data/X_test.csv')
y = np.ravel(pd.read_csv('../data/y.csv'))

In [74]:
import joblib
cb_2 = joblib.load('models/best_catboost.pkl')

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_tr, y, test_size=0.25, random_state=42)

y_pred_proba = cb_2.predict_proba(X_val)[:, 1]
y_pred_binary = (y_pred_proba > 0.5).astype(int)

errors = y_val != y_pred_binary
error_idx = np.where(errors)[0]

error_samples = X_val.iloc[error_idx]

error_samples.head(3)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
224752,-1.591114,0.63176,0.100333,0.860655,-0.294858,-1.612353,-0.613913,-0.754928,-0.836168,-0.564825,-0.830189,0.0,0.0,1.0,0.0,1.0
395208,2.042477,0.63176,-0.033216,-0.326939,-0.294858,1.369924,-0.613913,-0.754928,-0.836168,-0.564825,-0.830189,1.0,0.0,0.0,0.0,0.0
383269,-0.016558,0.63176,-0.166765,0.534066,-0.294858,0.114228,1.628894,-0.544063,-0.836168,-0.564825,-0.830189,0.0,0.0,1.0,0.0,1.0


In [108]:
df = pd.read_csv('../data/train.csv', index_col = 'id')
raw_err = pd.DataFrame(df, index = error_idx)
raw_err

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
6,59,1,4,130,246,0,2,152,0,0.8,2,2,3,Presence
12,41,0,3,108,230,0,2,142,0,0.6,2,1,3,Absence
14,42,1,4,108,244,0,0,150,1,1.6,2,0,7,Presence
22,57,0,3,124,204,0,2,166,0,0.0,1,0,3,Absence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157402,63,1,4,120,254,0,2,170,1,0.9,2,2,7,Presence
157413,77,1,3,115,235,0,2,147,0,0.9,1,0,7,Presence
157440,57,1,4,125,303,0,2,143,1,2.0,2,2,7,Presence
157452,56,1,3,132,213,0,2,157,1,0.4,2,0,7,Presence


In [109]:
error_dis = raw_err['Heart Disease'].value_counts()

print(f"err abence {error_dis['Absence']/len(raw_err)*100}%")
print(f"err presence {error_dis['Presence']/len(raw_err)*100}%")

err abence 55.08306401315416%
err presence 44.916935986845836%


In [110]:
num_cols = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']

for col in num_cols:
    err_mean = raw_err[col].mean()
    full_mean = df[col].mean()  
    diff = (err_mean - full_mean) / full_mean * 100 
    print(col, round(diff, 1), '%')

Age -0.0 %
BP 0.0 %
Cholesterol -0.0 %
Max HR 0.0 %
ST depression -0.7 %


In [100]:
cat_cols = ['Sex', 'Chest pain type', 'FBS over 120', 'EKG results', 
                    'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium']

for col in cat_cols:
    err= raw_err[col].value_counts(normalize=True).sort_index() * 100
    full = df[col].value_counts(normalize=True).sort_index() * 100
 
    print((err - full).round(1))

Sex
0   -0.0
1    0.0
Name: proportion, dtype: float64
Chest pain type
1    0.3
2   -0.2
3    0.3
4   -0.4
Name: proportion, dtype: float64
FBS over 120
0   -0.1
1    0.1
Name: proportion, dtype: float64
EKG results
0   -0.3
1    0.1
2    0.2
Name: proportion, dtype: float64
Exercise angina
0   -0.2
1    0.2
Name: proportion, dtype: float64
Slope of ST
1   -0.0
2   -0.1
3    0.1
Name: proportion, dtype: float64
Number of vessels fluro
0    0.1
1   -0.2
2   -0.2
3    0.3
Name: proportion, dtype: float64
Thallium
3   -0.2
6   -0.0
7    0.2
Name: proportion, dtype: float64


ошибки случайны, распределения полных данных и тех, на которых модель ошиблась различаются минимально

новые признаки:
1) puls norm - норма максимального пульса отностиельно возраста /числ
2) bp chol - нагрузка на сердце /числ
3) st - дисперсия учитывая наклон / числ
4) sick - болезнь без симптомов(нет боли, с экг все нормально, при нагрузке тоже без сиптомов) /бинарн
5) age fluro - возраст и сосуды  /числ
6) ekg tall - плохие экг и  таллинум - болезнь /бинарн
7) risk - группа риска(много  плохих показателей) /числ
8) st tall - высокая дисперсия и высокий таллинум /бинарн
9) fluro - чем хуже флюраграфия, тем сильно выше  /порядк
10) chol - холестерин учитывая возраст, т к нормы разные /числ

In [111]:
def new_pr(df):
    df['puls norm'] = df['Max HR'] - (220 - df['Age']) 
    df['bp chol'] = df['BP'] * df['Cholesterol']
    df['st'] = df['ST depression'] * df['Slope of ST']
    df['sick'] = (((df['Chest pain type'] == 4) | (df['Chest pain type'] == 3)) * (df['Exercise angina'] == 0) * (df['ST depression'] < 0.5)).astype(int)
    df['age fluro'] = df['Age'] * df['Number of vessels fluro']
    df['ekg tall'] = (df['EKG results'] == 2).astype(int) * (df['Thallium']== 7).astype(int)
    df['risk'] = (df['Age'] > 55).astype(int) + (df['Sex'] == 1).astype(int) + (df['BP'] > 130).astype(int) + (df['Cholesterol'] > 240).astype(int) +(df['FBS over 120'] == 1).astype(int) + (df['Number of vessels fluro'] > 0).astype(int)
    df['st thall'] = ((df['ST depression'] > 0.5).astype(int) * (df['Thallium'] == 7).astype(int))
    df['fluro'] = df['Number of vessels fluro'] ** 2
    df['chol'] = df['Cholesterol'] - (200 + (df['Age'] - 40) * 0.7)
    return df


In [112]:
df_test = pd.read_csv('../data/test.csv', index_col = 'id')
df_tr = new_pr(df)
df_test = new_pr(df_test)
df.head(10)

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,...,puls norm,bp chol,st,sick,age fluro,ekg tall,risk,st thall,fluro,chol
id,,,,,,,,,,,,,,,,,,,,,
0,58,1,4,152,239,0,0,158,1,3.6,...,-4,36328,7.2,0,116,0,4,1,4,26.4
1,52,1,1,125,325,0,2,171,0,0.0,...,3,40625,0.0,0,0,0,2,0,0,116.6
2,56,0,2,160,188,0,2,151,0,0.0,...,-13,30080,0.0,0,0,0,2,0,0,-23.2
3,44,0,3,134,229,0,2,150,0,1.0,...,-26,30686,2.0,0,0,0,1,0,0,26.2
4,58,1,4,140,234,0,2,125,1,3.8,...,-37,32760,7.6,0,174,0,4,0,9,21.4
5,38,1,4,138,283,0,0,147,1,1.6,...,-35,39054,3.2,0,76,0,4,1,4,84.4
6,59,1,4,130,246,0,2,152,0,0.8,...,-9,31980,1.6,0,118,0,4,0,4,32.7
7,60,0,3,120,245,0,0,151,0,1.2,...,-9,29400,1.2,0,0,0,2,0,0,31.0
8,48,0,4,140,212,0,2,125,0,0.0,...,-47,29680,0.0,1,0,0,1,0,0,6.4


In [113]:
df_test.head()

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,...,puls norm,bp chol,st,sick,age fluro,ekg tall,risk,st thall,fluro,chol
id,,,,,,,,,,,,,,,,,,,,,
630000,58,1,3,120,288,0,2,145,1,0.8,...,-17,34560,1.6,0,174,0,4,0,9,75.4
630001,55,0,2,120,209,0,0,172,0,0.0,...,7,25080,0.0,0,0,0,0,0,0,-1.5
630002,54,1,4,120,268,0,0,150,1,0.0,...,-16,32160,0.0,0,162,0,3,0,9,58.2
630003,44,0,3,112,177,0,0,168,0,0.9,...,-8,19824,0.9,0,0,0,0,0,0,-25.8
630004,43,1,1,138,267,0,0,163,0,1.8,...,-14,36846,3.6,0,0,0,3,1,0,64.9


In [90]:
num_cols, cat_cols

(['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression'],
 ['Sex',
  'Chest pain type',
  'FBS over 120',
  'EKG results',
  'Exercise angina',
  'Slope of ST',
  'Number of vessels fluro',
  'Thallium'])

slope st(порядок наклона) (преобразование в порядковый)

In [114]:
from sklearn.preprocessing import OrdinalEncoder

ord_enc = OrdinalEncoder(categories=[[1, 2, 3]])
df['Slope of ST'] = ord_enc.fit_transform(df[['Slope of ST']])
df_test['Slope of ST'] = ord_enc.transform(df_test[['Slope of ST']])

In [115]:
bin_cols = ['st thall', 'sick', 'ekg tall', 'Sex', 'Exercise angina', 'FBS over 120']
new_num = ['chol', 'fluro', 'risk', 'age fluro', 'st', 'bp chol', 'puls norm', 'Number of vessels fluro', 'Slope of ST']
num_cols.extend(new_num)

to_oh_cols = []

for col in df.columns:
    if col not in bin_cols and col not in num_cols and col != 'Heart Disease':
        to_oh_cols.append(col)
        
to_oh_cols

['Chest pain type', 'EKG results', 'Thallium']

In [116]:
num_cols, bin_cols

(['Age',
  'BP',
  'Cholesterol',
  'Max HR',
  'ST depression',
  'chol',
  'fluro',
  'risk',
  'age fluro',
  'st',
  'bp chol',
  'puls norm',
  'Number of vessels fluro',
  'Slope of ST'],
 ['st thall', 'sick', 'ekg tall', 'Sex', 'Exercise angina', 'FBS over 120'])

подготовка

In [120]:
df_tr

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,...,puls norm,bp chol,st,sick,age fluro,ekg tall,risk,st thall,fluro,chol
id,,,,,,,,,,,,,,,,,,,,,
0,58,1,4,152,239,0,0,158,1,3.6,...,-4,36328,7.2,0,116,0,4,1,4,26.4
1,52,1,1,125,325,0,2,171,0,0.0,...,3,40625,0.0,0,0,0,2,0,0,116.6
2,56,0,2,160,188,0,2,151,0,0.0,...,-13,30080,0.0,0,0,0,2,0,0,-23.2
3,44,0,3,134,229,0,2,150,0,1.0,...,-26,30686,2.0,0,0,0,1,0,0,26.2
4,58,1,4,140,234,0,2,125,1,3.8,...,-37,32760,7.6,0,174,0,4,0,9,21.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,56,0,1,110,226,0,0,132,0,0.0,...,-32,24860,0.0,0,0,0,1,0,0,14.8
629996,54,1,4,128,249,1,2,150,0,0.0,...,-16,31872,0.0,1,0,0,3,0,0,39.2
629997,67,1,4,130,275,0,0,149,0,0.0,...,-4,35750,0.0,1,134,0,4,0,4,56.1


In [121]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import pandas as pd

df_tr.drop(columns=['Heart Disease'], inplace=True)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),  
        ('bin', 'passthrough', bin_cols)        
        ('oh_cat', OneHotEncoder(drop='first', sparse_output=False), to_oh_cols)  
    ])

X_tr = preprocessor.fit_transform(df_tr)  
X_test = preprocessor.transform(df_test)

X_tr = pd.DataFrame(X_tr, index=df_tr.index)
X_test = pd.DataFrame(X_test, index=df_test.index)

X_tr.to_csv('../data/X_tr_new.csv', index = False)
X_test.to_csv('../data/X_test_new.csv', index = False)


preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),  
        ('bin', 'passthrough', bin_cols),        
        ('oh_cat', OneHotEncoder(drop='first', sparse_output=False), to_oh_cols)  
    ])

X_tr = preprocessor.fit_transform(df_tr)  
X_test = preprocessor.transform(df_test)

X_tr = pd.DataFrame(X_tr, index=df_tr.index)
X_test = pd.DataFrame(X_test, index=df_test.index)

X_tr.to_csv('../data/X_tr_new.csv', index = False)
X_test.to_csv('../data/X_test_new.csv', index = False)